# Ejercicio 7 - Análisis y comparación de los lagos

Consolida en una sola tabla lo que ya calcularon los ejercicios 4, 5, 6,
8.1 y 8.2 para cada lago (promedio temporal, frecuencia sobre el umbral,
extensión de floración, persistencia espacial y correlaciones), para
comparar Atitlán y Amatitlán con métricas comunes en vez de una sola cifra
aislada.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src.comparacion_lagos import build_comparison_table, write_comparison_table
from src.config import LAGOS, UMBRAL_CIANOBACTERIA_ALTO_UGL

print('Umbral de "valor alto" (fijado en el ejercicio 5, antes de este análisis):',
      UMBRAL_CIANOBACTERIA_ALTO_UGL, 'µg/L')


Umbral de "valor alto" (fijado en el ejercicio 5, antes de este análisis): 10.0 µg/L


## 1. Tabla comparativa

Cada fila resume un lago con las mismas métricas: promedio general de
cianobacteria, cuántas fechas superan el umbral acordado, qué tan extensa
fue la floración en promedio y en su peor fecha, qué porcentaje del área
tuvo cianobacteria alta alguna vez frente a de forma persistente (del
ejercicio 8.2), y la correlación mediana con NDVI/NDWI (del ejercicio 6).

`n_fechas_calculado` cuenta solo fechas sin advertencia de calidad;
`n_fechas_total` son las 11 oficiales. La diferencia entre ambas columnas
ya es, en sí misma, información relevante para comparar los dos lagos.

In [2]:
tabla = build_comparison_table()
write_comparison_table(tabla)
df = pd.DataFrame(tabla).set_index('lago')
display(df.T)


lago,amatitlan,atitlan
n_fechas_calculado,10,5
n_fechas_total,11,11
cyano_promedio_general,6.2906,1.1535
cyano_mediana_general,5.7475,1.1416
frecuencia_fechas_sobre_umbral,1,0
pct_fechas_sobre_umbral,9.09,0.0
porcentaje_alto_promedio,10.9373,0.0264
porcentaje_alto_maximo,54.3977,0.1042
fecha_porcentaje_alto_maximo,2026-06-19,2026-07-22
pct_area_alguna_vez_alta,72.4,0.17


## 2. Intensidad y frecuencia: lo que dicen los números

No se compara con un solo máximo aislado: se usan promedio, mediana,
frecuencia sobre el umbral, extensión promedio/máxima y persistencia a la
vez, tal como pide la planificación.

In [3]:
resumen = df[[
    'cyano_promedio_general', 'frecuencia_fechas_sobre_umbral', 'pct_fechas_sobre_umbral',
    'porcentaje_alto_promedio', 'porcentaje_alto_maximo', 'pct_area_alguna_vez_alta',
    'pct_area_persistente', 'tendencia_temporal',
]]
display(resumen)

amatitlan = df.loc['amatitlan']
atitlan = df.loc['atitlan']
razon = amatitlan['cyano_promedio_general'] / atitlan['cyano_promedio_general']
print(f"Amatitlán promedia {razon:.1f}x la cianobacteria de Atitlán en el período estudiado.")
print(f"Amatitlán: {amatitlan['n_fechas_calculado']}/{amatitlan['n_fechas_total']} fechas sin advertencia de calidad.")
print(f"Atitlán:   {atitlan['n_fechas_calculado']}/{atitlan['n_fechas_total']} fechas sin advertencia de calidad.")


,cyano_promedio_general,frecuencia_fechas_sobre_umbral,pct_fechas_sobre_umbral,porcentaje_alto_promedio,porcentaje_alto_maximo,pct_area_alguna_vez_alta,pct_area_persistente,tendencia_temporal
lago,,,,,,,,
amatitlan,6.2906,1,9.09,10.9373,54.3977,72.40,0.150,creciente
atitlan,1.1535,0,0.00,0.0264,0.1042,0.17,0.006,estable


Amatitlán promedia 5.5x la cianobacteria de Atitlán en el período estudiado.
Amatitlán: 10/11 fechas sin advertencia de calidad.
Atitlán:   5/11 fechas sin advertencia de calidad.


## 3. Posibles diferencias entre ambos lagos

**Lo que miden los datos de este laboratorio** (evidencia directa, tabla de
arriba): Amatitlán tiene un promedio de cianobacteria varias veces mayor
que Atitlán, la única fecha que supera el umbral de 10 µg/L es de
Amatitlán, su extensión de floración promedio y máxima son mucho mayores,
y su tendencia en el período es creciente mientras que Atitlán se mantiene
estable en un nivel bajo.

**Contexto documentado, no medido por este laboratorio** (debe citarse y
no confundirse con una causa demostrada por los datos):

- Amatitlán está en una cuenca mucho más urbanizada y con más presión de
  aguas residuales e industriales que desembocan en el lago, en el área
  metropolitana de Ciudad de Guatemala; su autoridad de cuenca (AMSA,
  Autoridad para el Manejo Sustentable de la Cuenca del Lago de Amatitlán)
  documenta desde hace años un problema crónico de eutrofización y
  floraciones de cianobacterias asociado a esa carga de nutrientes.
- Atitlán es un lago volcánico mucho más profundo (uno de los más
  profundos de Centroamérica), con una cuenca de menor densidad urbana
  relativa (aunque en crecimiento) y menor carga histórica de aguas
  residuales sin tratar; su autoridad de cuenca es AMSCLAE (Autoridad para
  el Manejo Sustentable de la Cuenca del Lago de Atitlán y su Entorno).
  Atitlán tuvo un evento de floración importante documentado en 2009, fuera
  del período cubierto por este laboratorio (2025-2026).
- La profundidad y el volumen de un lago afectan cuánto tiempo permanecen
  los nutrientes cerca de la superficie: un lago más profundo y de mayor
  volumen como Atitlán diluye más la carga de nutrientes que uno más
  playo como Amatitlán, para una misma cantidad de nutrientes que entra.

Estos tres puntos son contexto ambiental ampliamente documentado sobre
ambos lagos, no una conclusión derivada de las imágenes de este
laboratorio; se citan para interpretar el patrón observado, no como prueba
de causalidad.

## 4. Limitación de confiabilidad de los datos entre lagos

Atitlán tuvo bastantes más fechas con advertencia de calidad
(`revisar_valores_atipicos`) que Amatitlán durante el período estudiado
(ver columna `n_fechas_calculado` arriba). Esto viene de una inestabilidad
numérica conocida del cálculo sobre agua muy profunda y oscura (ver
`codebook.md`, sección de NDVI/NDWI), consistente con que Atitlán es un
lago mucho más profundo que Amatitlán. Es decir: parte de la diferencia
observada también podría reflejar que los datos de Atitlán son, en
promedio, algo menos confiables que los de Amatitlán en este período —no
solo que Atitlán tenga menos cianobacteria. Ambas explicaciones son
compatibles y no se pueden separar completamente con los datos
disponibles.

## 5. Conclusión de la comparación

Con la evidencia de este laboratorio, Amatitlán mostró una proliferación de
cianobacteria consistentemente mayor y creciente durante el período
2025-2026, mientras que Atitlán se mantuvo en niveles bajos y estables.
Esto es consistente con la diferencia de presión urbana y profundidad entre
ambos lagos documentada por sus respectivas autoridades de cuenca, aunque
este laboratorio no midió directamente nutrientes, aguas residuales ni
profundidad — esa relación queda como contexto razonado, no como una causa
demostrada por los datos analizados aquí.